# 10 - Compare repeated k-mer evaluation and audit QAC metadata

This notebook evaluates the disk-light capped assembly k-mer baseline across the same ten lineage-aware train/dev/test partitions used for DNABERT2. It also inspects the supplementary `no_known_QAC_gene` field before any known-gene baseline is added.

## 1. Configuration

The repeated k-mer baseline reuses the compact public FASTA assemblies already downloaded. It does not save a large feature matrix.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
DNABERT_REPORT_PATH = PROJECT_ROOT / "reports" / "dnabert2_repeated_grouped_evaluation.json"
KMER_REPORT_PATH = PROJECT_ROOT / "reports" / "kmer_repeated_grouped_evaluation.json"
WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "gmeiner_2025" / "41598_2025_94321_MOESM1_ESM.xlsx"
SEEDS = "42,43,44,45,46,47,48,49,50,51"
RUN_REPEATED_KMER_EVALUATION = False

print("Project root:", PROJECT_ROOT)
print("Assembly manifest exists:", MANIFEST_PATH.exists())
print("DNABERT2 repeated report exists:", DNABERT_REPORT_PATH.exists())

## 2. Run the repeated capped k-mer baseline

Change `RUN_REPEATED_KMER_EVALUATION` to `True`. Each model is fit sequentially. Thresholds are selected using development isolates only, matching notebook 09.

In [ ]:
evaluation_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "evaluate_kmer_repeated_grouped_splits.py"),
    "--manifest",
    str(MANIFEST_PATH),
    "--seeds",
    SEEDS,
    "--output",
    str(KMER_REPORT_PATH),
]
print(" ".join(evaluation_command))
if RUN_REPEATED_KMER_EVALUATION:
    subprocess.run(evaluation_command, cwd=PROJECT_ROOT, check=True)
if KMER_REPORT_PATH.exists():
    kmer_report = json.loads(KMER_REPORT_PATH.read_text())
    print(json.dumps(kmer_report["summary"], indent=2))
    display(pd.DataFrame(kmer_report["evaluations"]))

## 3. Compare repeated held-out performance

This is a feasibility comparison, not a final publication-grade validation. Some test partitions contain only a few tolerant isolates.

In [ ]:
reports = {
    "DNABERT2 frozen embeddings": json.loads(DNABERT_REPORT_PATH.read_text()),
    "Capped assembly k-mer TF-IDF": json.loads(KMER_REPORT_PATH.read_text()),
}
comparison_rows = []
for model_name, report in reports.items():
    row = {"model": model_name}
    for metric, values in report["summary"].items():
        row[f"{metric}_mean"] = values["mean"]
        row[f"{metric}_std"] = values["std"]
    comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))

## 4. Audit the supplementary QAC flag

The paper states that 20 isolates were tolerant to BC in laboratory experiments although no known QAC resistance gene was observed. The supplementary `no_known_QAC_gene` plus sign identifies that exception set. It must **not** be interpreted as a known-gene-positive label.

In [ ]:
supplementary = pd.read_excel(WORKBOOK_PATH, sheet_name="Tables S1", header=1)
qac_flag = supplementary["no_known_QAC_gene"].fillna("").astype(str).str.strip()
bc_mic = pd.to_numeric(supplementary["BC MIC"], errors="coerce")
audit = pd.DataFrame({"no_known_QAC_gene": qac_flag.replace("", "<missing>"), "bc_mic_mg_l": bc_mic})
print("Rows marked as tolerant despite no known QAC gene:", int((qac_flag == "+").sum()))
display(pd.crosstab(audit["no_known_QAC_gene"], audit["bc_mic_mg_l"], margins=True))

## 5. Next step

Build an interpretable assembly-screening baseline for known determinants such as `bcrABC`, `emrC`, `emrE`, and `qacH`, then evaluate that baseline under the same grouped partitions. Keep the biological claim narrow: this project predicts low-level BC MIC tolerance, not survival after use-level sanitizer exposure.